# [SK 02 - Chat Completion OpenAI Client from SK SDK (WITH kernel and plugins)](https://learn.microsoft.com/en-us/semantic-kernel/get-started/quick-start-guide?pivots=programming-language-python#writing-your-first-console-app)

# Constants and Libraries

In [1]:
import os
from dotenv import load_dotenv # requires python-dotenv

load_dotenv("./../config/credentials_my.env")

service_id = "azure_chat_completion_service"
plugin_name="Lights"

print(f"os.environ['AZURE_OPENAI_ENDPOINT']: {os.environ['AZURE_OPENAI_ENDPOINT']}")

os.environ['AZURE_OPENAI_ENDPOINT']: https://mmoaiswc-01.openai.azure.com/


# Set the logging level for  semantic_kernel.kernel to DEBUG
One of the main benefits of using Semantic Kernel is that it supports enterprise-grade services.<br/>
In this sample, we add the logging service to the kernel to help debug the AI agent.

In [2]:
import logging

logging.basicConfig(
    format="[%(asctime)s - %(name)s:%(lineno)d - %(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S")
    
logging.getLogger("kernel").setLevel(logging.DEBUG)

# Create an `AzureChatCompletion` object e.g. the `assistant` from SK library

In [3]:
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion

azure_chat_completion=AzureChatCompletion(service_id=service_id)
azure_chat_completion

AzureChatCompletion(ai_model_id='gpt-4o-for-apim', service_id='azure_chat_completion_service', client=<openai.lib.azure.AsyncAzureOpenAI object at 0x0000021130A7DCA0>, ai_model_type=<OpenAIModelTypes.CHAT: 'chat'>, prompt_tokens=0, completion_tokens=0, total_tokens=0)

# Initialize an empty `Kernel`

In [4]:
from semantic_kernel import Kernel

kernel = Kernel()
kernel

Kernel(retry_mechanism=PassThroughWithoutRetry(), services={}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x000002113F131580>, plugins={}, function_invocation_filters=[], prompt_rendering_filters=[], auto_function_invocation_filters=[])

# Add AzureChatCompletion assistant as a service to the Kernel

In [5]:
kernel.add_service(azure_chat_completion)

kernel 

Kernel(retry_mechanism=PassThroughWithoutRetry(), services={'azure_chat_completion_service': AzureChatCompletion(ai_model_id='gpt-4o-for-apim', service_id='azure_chat_completion_service', client=<openai.lib.azure.AsyncAzureOpenAI object at 0x0000021130A7DCA0>, ai_model_type=<OpenAIModelTypes.CHAT: 'chat'>, prompt_tokens=0, completion_tokens=0, total_tokens=0)}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x000002113F131580>, plugins={}, function_invocation_filters=[], prompt_rendering_filters=[], auto_function_invocation_filters=[])

# Create a `Class` e.g. a `Plugin`

In [7]:
# First, we define the plugin through its class...

class LightsPlugin:
    from typing import Annotated
    from semantic_kernel.functions import kernel_function
    lights = [
        {"id": 0, "name": "Table Lamp", "is_on": False},
        {"id": 1, "name": "Porch light", "is_on": False},
        {"id": 2, "name": "Chandelier", "is_on": True},
    ]

    @kernel_function(
        name="get_lights", # <<<=== DIFFERENT FROM THE FUNCTION NAME <get_state>, which will be ignored
        description="Gets a list of lights and their current state",
    )
    def get_state(
        self,
    ) -> Annotated[str, "the output is a string"]:
        """Gets a list of lights and their current state."""
        return self.lights

    @kernel_function(
        name="change_state",
        description="Changes the state of the light",
    )
    def change_state(
        self,
        id: int,
        is_on: bool,
    ) -> Annotated[str, "the output is a string"]:
        """Changes the state of the light."""
        for light in self.lights:
            if light["id"] == id:
                light["is_on"] = is_on
                return light
        return None

# Add the (***native***) Plugin to the Kernel

In [8]:
# ...then, we add the plugin to the kernel, using a new plugin name

kernel.add_plugin(
    LightsPlugin(),
    plugin_name=plugin_name,
)

KernelPlugin(name='Lights', description=None, functions={'change_state': KernelFunctionFromMethod(metadata=KernelFunctionMetadata(name='change_state', plugin_name='Lights', description='Changes the state of the light', parameters=[KernelParameterMetadata(name='id', description=None, default_value=None, type_='int', is_required=True, type_object=<class 'int'>, schema_data={'type': 'integer'}, include_in_function_choices=True), KernelParameterMetadata(name='is_on', description=None, default_value=None, type_='bool', is_required=True, type_object=<class 'bool'>, schema_data={'type': 'boolean'}, include_in_function_choices=True)], is_prompt=False, is_asynchronous=False, return_parameter=KernelParameterMetadata(name='return', description='the output is a string', default_value=None, type_='str', is_required=True, type_object=<class 'str'>, schema_data={'type': 'string', 'description': 'the output is a string'}, include_in_function_choices=True), additional_properties={}), invocation_duratio

# Just for testing: extract Services and Plugins from the kernel

In [9]:
print (f"kernel.services:\n{kernel.services}\n") # or, kernel.get_service(service_id=service_id)
print (f"kernel.plugins:\n{kernel.plugins}\n") # or, kernel.get_plugin(plugin_name=plugin_name)

kernel.services:
{'azure_chat_completion_service': AzureChatCompletion(ai_model_id='gpt-4o-for-apim', service_id='azure_chat_completion_service', client=<openai.lib.azure.AsyncAzureOpenAI object at 0x0000021130A7DCA0>, ai_model_type=<OpenAIModelTypes.CHAT: 'chat'>, prompt_tokens=0, completion_tokens=0, total_tokens=0)}

kernel.plugins:
{'Lights': KernelPlugin(name='Lights', description=None, functions={'change_state': KernelFunctionFromMethod(metadata=KernelFunctionMetadata(name='change_state', plugin_name='Lights', description='Changes the state of the light', parameters=[KernelParameterMetadata(name='id', description=None, default_value=None, type_='int', is_required=True, type_object=<class 'int'>, schema_data={'type': 'integer'}, include_in_function_choices=True), KernelParameterMetadata(name='is_on', description=None, default_value=None, type_='bool', is_required=True, type_object=<class 'bool'>, schema_data={'type': 'boolean'}, include_in_function_choices=True)], is_prompt=False,

# Enable planning with Function Calling set as Auto()
## Note that `function_choice_behavior=FunctionChoiceBehavior` now

In [10]:
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior
from semantic_kernel.connectors.ai.open_ai.prompt_execution_settings.azure_chat_prompt_execution_settings import AzureChatPromptExecutionSettings

execution_settings = AzureChatPromptExecutionSettings()
execution_settings.function_choice_behavior= FunctionChoiceBehavior.Auto() # Auto(), Required() or NoneInvoke()
execution_settings

AzureChatPromptExecutionSettings(service_id=None, extension_data={}, function_choice_behavior=FunctionChoiceBehavior(enable_kernel_functions=True, maximum_auto_invoke_attempts=5, filters=None, type_=<FunctionChoiceType.AUTO: 'auto'>), ai_model_id=None, frequency_penalty=None, logit_bias=None, max_tokens=None, number_of_responses=None, presence_penalty=None, seed=None, stop=None, stream=False, temperature=None, top_p=None, user=None, store=None, metadata=None, response_format=None, function_call=None, functions=None, messages=None, function_call_behavior=None, parallel_tool_calls=True, tools=None, tool_choice=None, structured_json_response=False, stream_options=None, extra_body=None)

# Create a new User message that can leverage the added plug-in
This message requires to run the plugin we built above

In [11]:
from semantic_kernel.contents.chat_history import ChatHistory

# Create a blank history of the conversation
history = ChatHistory() # initially blank

# Add user input to the history
history.add_user_message("Toggle the status of my second light.")

history

ChatHistory(messages=[ChatMessageContent(inner_content=None, ai_model_id=None, metadata={}, content_type='message', role=<AuthorRole.USER: 'user'>, name=None, items=[TextContent(inner_content=None, ai_model_id=None, metadata={}, content_type='text', text='Toggle the status of my second light.', encoding=None)], encoding=None, finish_reason=None)])

# Put all togehter to invoke AzureChatCompletion assistant

In [12]:
response = await azure_chat_completion.get_chat_message_contents(
    chat_history=history,
    settings=execution_settings,
    kernel=kernel)

for r in response:
    history.add_message(r)

#  Print the results
print(f"\nAssistant's response: {response[0].content}\n")

history


Assistant's response: The status of your second light, the "Porch light," has been toggled successfully. It is now turned on.



ChatHistory(messages=[ChatMessageContent(inner_content=None, ai_model_id=None, metadata={}, content_type='message', role=<AuthorRole.USER: 'user'>, name=None, items=[TextContent(inner_content=None, ai_model_id=None, metadata={}, content_type='text', text='Toggle the status of my second light.', encoding=None)], encoding=None, finish_reason=None), ChatMessageContent(inner_content=ChatCompletion(id='chatcmpl-AsSYWItQjv7K0vCagKkEnQxGeBYfV', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_w5L4d7aQfbhEMixtavDW3KLb', function=Function(arguments='{}', name='Lights-get_lights'), type='function')]), content_filter_results={})], created=1737542844, model='gpt-4o-2024-05-13', object='chat.completion', service_tier=None, system_fingerprint='fp_f3927aa00d', usage=CompletionUsage(completion_tokens=13, prompt_tokens=78, tota

# Extract Plugin invokation from the history

In [13]:
i=0
for cmc in history.messages: # ChatMessageContent
    if not cmc.inner_content is None:
        for choice in cmc.inner_content.choices:            
            if choice.message.tool_calls is None:
                i += 1
                print(f"{i} - choice.finish_reason: {choice.finish_reason}")
            else:
                for tc in choice.message.tool_calls:
                    i += 1
                    print (f"{i} - Call {tc.function.name}({tc.function.arguments})")

1 - Call Lights-get_lights({})
2 - Call Lights-change_state({"id":1,"is_on":true})
3 - choice.finish_reason: stop


# Additional tests. Run multiple times to toggle the first light only

In [16]:
history = ChatHistory() # initially blank
history.add_user_message("Toggle the first light and give me the status of all my lights.")

response = await azure_chat_completion.get_chat_message_contents(
    chat_history=history,
    settings=execution_settings,
    kernel=kernel)

for r in response:
    history.add_message(r)

#  Print the results
print(f"\nAssistant's response: {response[0].content}\n")

print("\nPlugin invokation:")

i=0
for cmc in history.messages: # ChatMessageContent
    if not cmc.inner_content is None:
        for choice in cmc.inner_content.choices:            
            if choice.message.tool_calls is None:
                i += 1
                print(f"{i} - choice.finish_reason: {choice.finish_reason}")
            else:
                for tc in choice.message.tool_calls:
                    i += 1
                    print (f"{i} - Call {tc.function.name}({tc.function.arguments})")

print("\nHistory:")
history


Assistant's response: Here is the status of all your lights:

1. **Table Lamp** - On
2. **Porch Light** - On
3. **Chandelier** - On


Plugin invokation:
1 - Call Lights-get_lights({})
2 - Call Lights-change_state({"id": 0, "is_on": true})
3 - Call Lights-get_lights({})
4 - choice.finish_reason: stop

History:


ChatHistory(messages=[ChatMessageContent(inner_content=None, ai_model_id=None, metadata={}, content_type='message', role=<AuthorRole.USER: 'user'>, name=None, items=[TextContent(inner_content=None, ai_model_id=None, metadata={}, content_type='text', text='Toggle the first light and give me the status of all my lights.', encoding=None)], encoding=None, finish_reason=None), ChatMessageContent(inner_content=ChatCompletion(id='chatcmpl-AsSaCu4jlNvAcWNox0MHuzGSXicV4', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_YdqrrPDtYamb9hmWetn4NH5U', function=Function(arguments='{}', name='Lights-get_lights'), type='function')]), content_filter_results={})], created=1737542948, model='gpt-4o-2024-05-13', object='chat.completion', service_tier=None, system_fingerprint='fp_f3927aa00d', usage=CompletionUsage(completion_tokens=